# Hebbian Trace: Persistent Memory for Frozen LLMs

**One-shot factual override, multi-hop reasoning, and 1000-fact capacity -- zero training.**

This notebook demonstrates [Hebbian Trace Memory](https://github.com/cnails/hebbian-trace-memory), a lightweight external memory module (~1.1M parameters) that attaches to any frozen language model.

| Demo | What it shows |
|------|---------------|
| 1. Override GPT-2's knowledge | Trace injection overrides built-in predictions |
| 2. One-line fact storage | `write_fact_direct()` API |
| 3. Capacity stress test | Baseline collapses, trace banks hold 99%+ |
| 4. Multi-hop reasoning | Chain trace lookups for 2-hop QA |
| 5. Interactive sandbox | Store and retrieve your own facts |
| Bonus: Phi-2 (2.7B) | Same module, 20x larger model, zero training |

**Works on CPU.** GPU (free T4) only needed for the Phi-2 bonus.

In [ ]:
# @title Install Dependencies { display-mode: "form" }
!pip install -q transformers matplotlib
!git clone -q https://github.com/cnails/hebbian-trace-memory.git 2>/dev/null || true
import os
os.chdir('/content/hebbian-trace-memory' if os.path.exists('/content') else os.path.join(os.getcwd(), 'hebbian-trace-memory') if not os.getcwd().endswith('hebbian-trace-memory') else os.getcwd())
!pip install -q -e .

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from hebbian_trace.gpt2_trace import GPT2WithTrace
from hebbian_trace.gpt2_tasks import get_linking_bpe_ids

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained('gpt2')

model = GPT2WithTrace(
    n_trace_heads=8, d_trace=64,
    alpha=0.5, trace_lr=1.0, trace_decay=0.99,
    device=device,
)
model.enable_pattern_separation(expand_factor=8, top_k=16, seed=0)
model.set_linking_token_ids(get_linking_bpe_ids(tokenizer))
model.set_bank_mode(16)

trace_params = sum(p.numel() for p in model.trace.parameters())
total_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 Small + Hebbian Trace on {device}")
print(f"  Trace module: {trace_params:,} params")
print(f"  Total: {total_params / 1e6:.1f}M params (GPT-2 frozen)")

---
## 1. Override GPT-2's Built-in Knowledge

GPT-2 "knows" that the capital of France is Paris. Can we override that with a single write?

We write a counterfactual fact to the trace, then show how the trace's **logit injection** overrides GPT-2's built-in predictions. The frozen model weights are never modified.

In [ ]:
counterfacts = [
    {
        "prompt": "The capital of France is",
        "fact": "The capital of France is Berlin",
        "fake": "Berlin", "real": "Paris",
        "candidates": ["Paris", "Berlin", "London", "Tokyo", "Madrid", "Rome", "Moscow"],
    },
    {
        "prompt": "The color of the sky is",
        "fact": "The color of the sky is green",
        "fake": "green", "real": "blue",
        "candidates": ["blue", "green", "red", "white", "gray", "black", "yellow"],
    },
]

results = []
for cf in counterfacts:
    cand_ids = [tokenizer.encode(" " + c, add_special_tokens=False)[0]
                for c in cf["candidates"]]

    # --- Baseline: GPT-2 without trace ---
    model.reset_traces()
    model.set_trace_mode(use=False, update=False)
    q_tensor = torch.tensor(
        [tokenizer.encode(cf["prompt"])], device=device)
    with torch.no_grad():
        bl = model(q_tensor)[0, -1, :].float()
    bl_probs = torch.softmax(bl[cand_ids], dim=0)

    # --- Write counterfactual via forward pass ---
    model.reset_traces()
    model.set_trace_mode(use=False, update=True)
    fact_tensor = torch.tensor(
        [tokenizer.encode(cf["fact"])], device=device)
    with torch.no_grad():
        model(fact_tensor)

    # --- Query with trace ---
    model.set_trace_mode(use=True, update=False)
    with torch.no_grad():
        tr = model(q_tensor)[0, -1, :].float()
    tr_probs = torch.softmax(tr[cand_ids], dim=0)

    results.append(dict(
        prompt=cf["prompt"], candidates=cf["candidates"],
        bl=bl_probs.cpu().numpy(), tr=tr_probs.cpu().numpy(),
        real=cf["real"], fake=cf["fake"],
    ))

    ri, fi = cf["candidates"].index(cf["real"]), cf["candidates"].index(cf["fake"])
    print(f'{cf["prompt"]}')
    print(f'  GPT-2:    {cf["real"]} {bl_probs[ri]:.1%}  |  {cf["fake"]} {bl_probs[fi]:.1%}')
    print(f'  + Trace:  {cf["real"]} {tr_probs[ri]:.1%}  |  {cf["fake"]} {tr_probs[fi]:.1%}')
    print()

In [ ]:
# @title The Logit Battle { display-mode: "form" }
fig, axes = plt.subplots(1, len(results), figsize=(8 * len(results), 5))
if len(results) == 1:
    axes = [axes]

for ax, r in zip(axes, results):
    x = np.arange(len(r["candidates"]))
    w = 0.35
    ax.bar(x - w / 2, r["bl"] * 100, w, label="GPT-2 (frozen)",
           color="#4A90D9", alpha=0.9)
    bars = ax.bar(x + w / 2, r["tr"] * 100, w, label="+ Hebbian Trace",
                  color="#E74C3C", alpha=0.9)

    # Highlight the injected answer
    fi = r["candidates"].index(r["fake"])
    bars[fi].set_edgecolor("black")
    bars[fi].set_linewidth(2)

    ax.set_xticks(x)
    ax.set_xticklabels(r["candidates"], rotation=45, ha="right", fontsize=10)
    ax.set_ylabel("Probability (%)")
    ax.set_title(r["prompt"] + " ___", fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_ylim(0, 105)

plt.suptitle(
    "Hebbian Trace overrides GPT-2's built-in knowledge",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

---
## 2. One-Line Fact Storage

The `write_fact_direct(concept_id, entity_id)` API stores a fact in ~0.4ms.
No templates, no forward pass -- just a Hebbian outer-product update.

Retrieval via `retrieve_direct_best_bank()` scans all trace banks and picks the answer with the highest confidence.

In [ ]:
model.reset_traces()

facts = [
    ("name", "Alice"),
    ("city", "Mars"),
    ("color", "purple"),
    ("food", "pizza"),
    ("pet", "dragon"),
]

# Write
print("Writing facts:")
for concept, entity in facts:
    cid = tokenizer.encode(" " + concept, add_special_tokens=False)[0]
    eid = tokenizer.encode(" " + entity, add_special_tokens=False)[0]
    model.write_fact_direct(cid, eid)
    print(f"  {concept} -> {entity}")

# Candidate pool (facts + distractors)
pool = ["Alice", "Bob", "Carol", "Mars", "Earth", "Venus",
        "purple", "blue", "red", "pizza", "sushi", "pasta",
        "dragon", "cat", "dog"]
pool_ids = [tokenizer.encode(" " + e, add_special_tokens=False)[0] for e in pool]

# Query
print("\nQuerying:")
correct = 0
for concept, expected in facts:
    cid = tokenizer.encode(" " + concept, add_special_tokens=False)[0]
    pred_id = model.retrieve_direct_best_bank(cid, pool_ids)
    pred = tokenizer.decode([pred_id]).strip()
    ok = pred == expected
    correct += ok
    print(f"  {concept}? -> {pred}  {'[correct]' if ok else f'[expected {expected}]'}")

print(f"\nAccuracy: {correct}/{len(facts)}")

---
## 3. Capacity Stress Test

What happens when we store 10, 50, or 100 facts?

Without banks, trace interference causes accuracy to collapse. **Hashed trace banks** route each fact to a separate trace matrix via sparse activation patterns -- zero new parameters, zero training.

In [ ]:
import random, time

# Build pools of single-token concepts and entities
concept_words = [
    "name", "city", "color", "food", "pet", "hobby", "car", "book",
    "song", "film", "game", "tool", "plant", "metal", "sport",
    "bird", "fish", "fruit", "tree", "river", "lake",
    "king", "queen", "hero", "poet", "chef", "pilot",
    "doctor", "nurse", "judge", "mayor", "knight",
    "castle", "bridge", "tower", "temple", "palace", "church",
    "winter", "summer", "spring", "autumn", "morning", "evening",
    "crystal", "diamond", "silver", "copper", "iron", "steel",
    "cotton", "silk", "marble", "glass", "paper",
    "tiger", "eagle", "whale", "shark", "wolf", "bear", "fox",
    "lion", "snake", "owl", "crow", "hawk", "deer", "horse",
    "maple", "cedar", "pine", "oak", "elm", "ash", "ivy",
    "storm", "flame", "frost", "thunder", "shadow", "crystal",
    "sword", "shield", "crown", "throne", "banner", "compass",
    "harbor", "valley", "canyon", "desert", "cliff", "reef",
    "violin", "flute", "drum", "piano", "guitar", "horn",
    "orbit", "comet", "nova", "pulse", "volt", "prism",
]

entity_words = [
    "Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace",
    "Paris", "London", "Tokyo", "Rome", "Berlin", "Madrid", "Moscow",
    "red", "blue", "green", "gold", "pink", "gray", "white",
    "Mars", "Venus", "Saturn", "Neptune", "Jupiter", "Mercury",
    "Jazz", "Rock", "Pop", "Folk", "Blues", "Soul", "Punk",
    "Python", "Java", "Ruby", "Rust", "Swift", "Perl",
    "Seven", "Eight", "Nine", "Three", "Four", "Five",
]

def _tok(words):
    """Filter to single-token words, return (word, token_id) pairs."""
    out, seen = [], set()
    for w in words:
        ids = tokenizer.encode(" " + w, add_special_tokens=False)
        if len(ids) == 1 and ids[0] not in seen:
            out.append((w, ids[0]))
            seen.add(ids[0])
    return out

concepts = _tok(concept_words)
entities = _tok(entity_words)
entity_ids = [eid for _, eid in entities]
print(f"Pool: {len(concepts)} concepts, {len(entities)} entities")

# Sweep
n_facts_list = [10, 20, 30, 50, 75, 100]
n_episodes = 10
configs = {"No banks (baseline)": 1, "16 banks": 16}
all_results = {name: [] for name in configs}

t0 = time.time()
for name, n_banks in configs.items():
    print(f"\n{name}:")
    for nf in n_facts_list:
        if nf > len(concepts):
            break
        hits = 0
        total = 0
        for ep in range(n_episodes):
            rng = random.Random(ep * 1000 + nf)
            c_sample = rng.sample(concepts, nf)
            e_sample = [rng.choice(entities) for _ in range(nf)]

            model.set_bank_mode(n_banks)
            model.reset_traces()

            for (_, cid), (_, eid) in zip(c_sample, e_sample):
                model.write_fact_direct(cid, eid)

            for (_, cid), (_, expected_eid) in zip(c_sample, e_sample):
                pred = model.retrieve_direct_best_bank(cid, entity_ids)
                hits += (pred == expected_eid)
                total += 1

        acc = hits / total
        all_results[name].append((nf, acc))
        print(f"  n={nf:3d}: {acc:.1%}")

print(f"\nDone in {time.time() - t0:.1f}s")

In [ ]:
# @title Capacity Curve { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 6))
styles = {
    "No banks (baseline)": ("#E74C3C", "o", "--"),
    "16 banks": ("#2ECC71", "s", "-"),
}

for name, res in all_results.items():
    ns, accs = zip(*res)
    color, marker, ls = styles[name]
    ax.plot(ns, [a * 100 for a in accs], marker=marker, linestyle=ls,
            label=name, color=color, linewidth=2.5, markersize=8)

ax.set_xlabel("Number of Facts Stored", fontsize=13)
ax.set_ylabel("Retrieval Accuracy (%)", fontsize=13)
ax.set_title("Capacity Scaling: Hashed Trace Banks vs Baseline",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=12, loc="lower left")
ax.set_ylim(0, 105)
ax.set_xlim(0, max(n_facts_list) + 5)
ax.grid(True, alpha=0.3)
ax.axhline(y=100, color="gray", linestyle=":", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Multi-Hop Reasoning

Chain two trace lookups: **concept -> bridge -> answer**.

Variant B from the paper: decode intermediate via `wte` argmax, re-encode as Q, query trace again. **Zero new parameters.**

```
Q(capital) -> T_v -> Paris -> Q(Paris) -> T_v -> France
```

In [ ]:
model.reset_traces()
model.set_bank_mode(16)

chains = [
    {"concept": "capital", "bridge": "Paris",  "answer": "France"},
    {"concept": "hero",    "bridge": "Alice", "answer": "Mars"},
    {"concept": "king",    "bridge": "Bob",   "answer": "Tokyo"},
]

# All tokens that can appear as intermediate or final answer
all_names = sorted({v for c in chains for v in [c["bridge"], c["answer"]]})
all_ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in all_names]

# Store chains
print("Storing chains:")
for c in chains:
    cid = tokenizer.encode(" " + c["concept"], add_special_tokens=False)[0]
    bid = tokenizer.encode(" " + c["bridge"], add_special_tokens=False)[0]
    aid = tokenizer.encode(" " + c["answer"], add_special_tokens=False)[0]

    model.write_fact_direct(cid, bid)   # hop 1: concept -> bridge
    model.write_fact_direct(bid, aid)   # hop 2: bridge -> answer
    print(f"  {c['concept']} -> {c['bridge']} -> {c['answer']}")

# Multi-hop retrieval
print("\nMulti-hop retrieval:")
for c in chains:
    cid = tokenizer.encode(" " + c["concept"], add_special_tokens=False)[0]

    # Hop 1
    hop1 = model.retrieve_direct(cid, all_ids)
    hop1_name = tokenizer.decode([hop1]).strip()

    # Hop 2 (feed hop1 output back as query)
    hop2 = model.retrieve_direct(hop1, all_ids)
    hop2_name = tokenizer.decode([hop2]).strip()

    ok = hop2_name == c["answer"]
    print(f"  {c['concept']} -> {hop1_name} -> {hop2_name}  "
          f"{'[correct]' if ok else f'[expected {c["answer"]}]'}")

---
## 5. Try It Yourself

Store your own facts and query them. Both concept and entity must be **single BPE tokens** (most short English words work).

In [ ]:
model.reset_traces()
model.set_bank_mode(16)
print("Trace memory cleared. Ready for your facts.")

In [ ]:
# @title Store a Fact { display-mode: "form" }
concept = "name" # @param {type:"string"}
entity = "Alice" # @param {type:"string"}

c_ids = tokenizer.encode(" " + concept, add_special_tokens=False)
e_ids = tokenizer.encode(" " + entity, add_special_tokens=False)

if len(c_ids) != 1:
    toks = [tokenizer.decode([i]) for i in c_ids]
    print(f"'{concept}' splits into {len(c_ids)} BPE tokens: {toks}")
    print("Try a shorter word (3-6 letters).")
elif len(e_ids) != 1:
    toks = [tokenizer.decode([i]) for i in e_ids]
    print(f"'{entity}' splits into {len(e_ids)} BPE tokens: {toks}")
    print("Try a shorter word (3-6 letters).")
else:
    model.write_fact_direct(c_ids[0], e_ids[0])
    print(f"Stored: {concept} -> {entity}")

In [ ]:
# @title Query a Fact { display-mode: "form" }
query_concept = "name" # @param {type:"string"}

q_ids = tokenizer.encode(" " + query_concept, add_special_tokens=False)
if len(q_ids) != 1:
    print(f"'{query_concept}' is multi-token. Try a shorter word.")
else:
    # Broad candidate pool
    _cands = [
        "Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace",
        "Paris", "London", "Tokyo", "Rome", "Berlin", "Mars", "Venus",
        "red", "blue", "green", "gold", "purple", "white", "black",
        "pizza", "sushi", "pasta", "rice", "bread", "cake",
        "dragon", "cat", "dog", "wolf", "eagle", "tiger", "bear",
        "Jazz", "Rock", "Pop", "Python", "Java", "Ruby", "Rust",
        "Seven", "Eight", "Nine", "Three", "Four", "Five", "Six",
    ]
    cand_ids = []
    cand_names = []
    for c in _cands:
        ids = tokenizer.encode(" " + c, add_special_tokens=False)
        if len(ids) == 1:
            cand_ids.append(ids[0])
            cand_names.append(c)

    pred_id = model.retrieve_direct_best_bank(q_ids[0], cand_ids)
    pred = tokenizer.decode([pred_id]).strip()
    print(f"{query_concept} -> {pred}")

---
## Bonus: Scaling to Phi-2 (2.7B)

The same Hebbian trace module works on larger models with zero training.

**Requires a GPU runtime** (Runtime -> Change runtime type -> T4).
First run downloads ~5GB of model weights.

In [ ]:
# @title Phi-2 Zero-Shot Transfer { display-mode: "form" }
if device != 'cuda':
    print("This cell requires a GPU runtime.")
    print("Go to Runtime -> Change runtime type -> T4 GPU, then re-run.")
else:
    print("Loading Phi-2 (2.7B)... ~1 min on first run.")

    phi_tok = AutoTokenizer.from_pretrained(
        "microsoft/phi-2", trust_remote_code=True)
    phi = GPT2WithTrace(
        n_trace_heads=8, d_trace=64,
        alpha=50.0,          # Phi-2 needs higher alpha
        trace_lr=1.0, trace_decay=0.99,
        model_name="microsoft/phi-2",
        torch_dtype=torch.float16,
        device=device,
    )
    phi.enable_pattern_separation(expand_factor=8, top_k=16, seed=0)
    phi.set_bank_mode(16)

    # Same API, larger model
    phi_facts = [("capital", "Berlin"), ("color", "green"), ("hero", "Alice")]
    cand_names = ["Paris", "Berlin", "London", "Tokyo",
                  "blue", "green", "red", "white",
                  "Alice", "Bob", "Carol", "Dave"]
    cand_ids = [phi_tok.encode(" " + c, add_special_tokens=False)[0]
                for c in cand_names
                if len(phi_tok.encode(" " + c, add_special_tokens=False)) == 1]

    for concept, entity in phi_facts:
        cid = phi_tok.encode(" " + concept, add_special_tokens=False)[0]
        eid = phi_tok.encode(" " + entity, add_special_tokens=False)[0]
        phi.write_fact_direct(cid, eid)

    print("\nPhi-2 (2.7B) + Hebbian Trace:")
    for concept, expected in phi_facts:
        cid = phi_tok.encode(" " + concept, add_special_tokens=False)[0]
        pred_id = phi.retrieve_direct_best_bank(cid, cand_ids)
        pred = phi_tok.decode([pred_id]).strip()
        ok = pred == expected
        print(f"  {concept} -> {pred}  {'[correct]' if ok else f'[expected {expected}]'}")

    print("\nSame module, same API, 20x larger model. Zero training.")

    del phi, phi_tok
    torch.cuda.empty_cache()

---

**Learn more:** [GitHub](https://github.com/cnails/hebbian-trace-memory) | [Architecture](https://github.com/cnails/hebbian-trace-memory/blob/main/ARCHITECTURE.md)

Key mechanisms: Pattern Separation (dentate gyrus), Dual Gating (ACh modulation), Reconsolidation Erasure, Hashed Trace Banks, T_auto (autoassociative completion).